# RAG по ТФКП: Qwen3.5-4B в Colab

Пять режимов: без контекста, один BM25-поиск, итоговый гибридный поиск с соседями,
без математического канала, без соседей; по 15 вопросов


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, zipfile, json, importlib.util, tempfile
ALLOW_CPU = False
RESUME = False
ARCHIVE_PATH = None  # Путь к ZIP или к папке Dataset; None — автоматический поиск
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
BASE.mkdir(parents=True, exist_ok=True)
ROOT = BASE / 'rag_solution'
def copy_source_tree(location):
    projects = [p for p in location.rglob('run.py') if (p.parent/'src/predict.py').is_file()]
    if len(projects) != 1:
        raise ValueError('Не найдена единственная папка проекта: '+str(projects))
    shutil.copytree(projects[0].parent, ROOT, dirs_exist_ok=True)
if not (ROOT / 'predict.py').exists():
    unpacked = None
    if ARCHIVE_PATH:
        supplied = Path(ARCHIVE_PATH)
        if not supplied.exists():
            raise FileNotFoundError('Не найден путь: '+str(supplied))
        if supplied.is_dir():
            candidates = list(supplied.rglob('*.zip'))
            if not candidates:
                unpacked = supplied
        else:
            candidates = [supplied]
    else:
        candidates = list(BASE.glob('RAG_TFKP_*.zip'))
        if not candidates and Path('/kaggle/input').exists():
            candidates = list(Path('/kaggle/input').rglob('RAG_TFKP_*.zip'))
            if not candidates:
                unpacked = Path('/kaggle/input')
        if not candidates and unpacked is None:
            from google.colab import files
            uploaded = files.upload()
            candidates = [Path(name).resolve() for name in uploaded if name.endswith('.zip')]
    if unpacked is not None:
        copy_source_tree(unpacked)
    else:
        if len(candidates) != 1:
            raise ValueError('Нужен один ZIP; укажите ARCHIVE_PATH: '+str(candidates))
        with tempfile.TemporaryDirectory(dir=BASE) as temporary:
            destination = Path(temporary).resolve()
            with zipfile.ZipFile(candidates[0]) as archive:
                if sum(e.file_size for e in archive.infolist()) > 100 * 1024**2:
                    raise ValueError('Архив кода/checkpoint превышает 100 МБ')
                for entry in archive.namelist():
                    if not (destination / entry).resolve().is_relative_to(destination):
                        raise ValueError('Небезопасный путь в архиве')
                archive.extractall(destination)
            copy_source_tree(destination)
for required in ('run.py', 'src/predict.py', 'configs/config.json', 'scripts/prepare_all.py'):
    if not (ROOT/required).is_file():
        raise FileNotFoundError('Неполная или старая структура проекта: '+required)
os.chdir(ROOT)
print('Проект:', ROOT, '; продолжение:', RESUME)


## 1. Проверка GPU


In [ ]:
probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
GPU = probe is not None and probe.returncode == 0
if GPU:
    print(probe.stdout)
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Используем одну GPU, в том числе на Kaggle T4 x2
elif not ALLOW_CPU:
    raise RuntimeError('GPU не обнаружена. Выберите GPU runtime и выполните ячейки снова; CPU здесь отключён.')
else:
    print('Явно выбран медленный режим CPU.')
free_gib = shutil.disk_usage(ROOT).free / 1024**3
print('Свободно на диске, GiB:', round(free_gib, 1))
if free_gib < 7:
    raise RuntimeError('Для весов, окружения и сборки нужно освободить место на диске.')


## 2. Защита от потери сессии

In [ ]:
SAVE_TO_DRIVE = (BASE == Path('/content'))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_dir = Path('/content/drive/MyDrive/RAG_TFKP_checkpoints')
else:
    checkpoint_dir = BASE / 'rag_private_checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
os.environ['RAG_CHECKPOINT_DIR'] = str(checkpoint_dir)
print('Checkpoint:', checkpoint_dir / 'RAG_TFKP_checkpoint.zip')


## 3. Python 3.11 и зависимости


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import json
import ctypes

os.chdir(ROOT)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'uv==0.13.4'],
    check=True
)

UV = shutil.which('uv')
if not UV:
    raise RuntimeError('uv установлен, но не найден в PATH.')

subprocess.run([UV, 'python', 'install', '3.11.17'], check=True)

PY = str(ROOT / '.venv/bin/python')
if not Path(PY).exists():
    subprocess.run(
        [UV, 'venv', '--python', '3.11.17', str(ROOT / '.venv')],
        check=True
    )

subprocess.run(
    [UV, 'pip', 'install', '--python', PY, '-r', 'requirements/core.txt'],
    check=True
)

env = dict(os.environ, CMAKE_BUILD_PARALLEL_LEVEL='2')

if GPU:
    nvcc = shutil.which('nvcc')
    if not nvcc and Path('/usr/local/cuda/bin/nvcc').exists():
        nvcc = '/usr/local/cuda/bin/nvcc'
        env['PATH'] = '/usr/local/cuda/bin:' + env['PATH']

    if not nvcc:
        raise RuntimeError('В GPU-сессии не найден CUDA compiler nvcc.')

    caps = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        text=True
    )
    arch = caps.splitlines()[0].strip().replace('.', '')
    if not arch.isdigit():
        raise RuntimeError('Не определена compute capability GPU: ' + caps)

    env['CMAKE_ARGS'] = (
        f'-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={arch}'
    )
    env['CUDACXX'] = nvcc

    # CUDA driver discovery
    drivers = []

    try:
        cuda = ctypes.CDLL('libcuda.so.1')
    except OSError:
        cuda = None

    if cuda is not None:
        for line in Path('/proc/self/maps').read_text().splitlines():
            fields = line.split(maxsplit=5)
            if len(fields) == 6:
                path = Path(fields[5].strip())
                if path.name.startswith('libcuda.so'):
                    drivers.append(path)

    for folder in (
        '/usr/lib64-nvidia',
        '/usr/local/nvidia/lib64',
        '/usr/lib/x86_64-linux-gnu',
        '/usr/lib/x86_64-linux-gnu/nvidia/current',
        '/usr/lib64',
        '/usr/lib/wsl/lib',
    ):
        drivers.extend(Path(folder).glob('libcuda.so*'))

    drivers = [
        p.resolve() for p in drivers
        if p.is_file() and 'stubs' not in p.resolve().parts
    ]

    if not drivers:
        raise RuntimeError(
            'Драйвер CUDA не найден. Проверьте nvidia-smi '
            'в отдельной ячейке.'
        )

    cuda_driver = drivers[0]
    cuda = ctypes.CDLL(str(cuda_driver))
    cuda.cuInit.argtypes = [ctypes.c_uint]
    cuda.cuInit.restype = ctypes.c_int
    status = cuda.cuInit(0)

    if status != 0:
        raise RuntimeError(
            f'Драйвер загружен, но cuInit вернул {status}. '
            'Проверьте nvidia-smi в отдельной ячейке.'
        )

    driver_dir = str(cuda_driver.parent)
    env['CMAKE_ARGS'] += (
        f' -DCUDA_cuda_driver_LIBRARY={cuda_driver}'
    )
    env['CMAKE_LIBRARY_PATH'] = (
        driver_dir + os.pathsep + env.get('CMAKE_LIBRARY_PATH', '')
    )
    os.environ['LD_LIBRARY_PATH'] = (
        driver_dir + os.pathsep + os.environ.get('LD_LIBRARY_PATH', '')
    )
    env['LD_LIBRARY_PATH'] = os.environ['LD_LIBRARY_PATH']

    print('CUDA driver:', cuda_driver, '; cuInit:', status)

else:
    env['CMAKE_ARGS'] = '-DGGML_CUDA=OFF'

probe_code = (
    'import json,llama_cpp;'
    'print(json.dumps({'
    '"version":llama_cpp.__version__,'
    '"gpu":bool(llama_cpp.llama_supports_gpu_offload())'
    '}))'
)

installed = subprocess.run(
    [PY, '-c', probe_code],
    capture_output=True,
    text=True
)

ready = False
if installed.returncode == 0:
    state = json.loads(installed.stdout.strip().splitlines()[-1])
    ready = (
        state['version'] == '0.3.36'
        and (not GPU or state['gpu'])
    )

if not ready:
    print('Сборка движка:', env['CMAKE_ARGS'], flush=True)
    subprocess.run(
        [
            UV, 'pip', 'install', '--python', PY,
            '--reinstall', '--no-deps',
            '--no-binary', 'llama-cpp-python',
            'llama-cpp-python==0.3.36'
        ],
        env=env,
        check=True
    )

subprocess.run([PY, '-c', probe_code], check=True)
subprocess.run([UV, 'pip', 'check', '--python', PY], check=True)

def run(*args):
    if not args:
        raise ValueError('Не указана команда')
    if str(args[0]).startswith('-'):
        command = [PY, '-u', *map(str, args)]
    else:
        command = [PY, '-u', str(ROOT/'run.py'), *map(str, args)]
    subprocess.run(command, cwd=ROOT, check=True)

print('Установка завершена. Можно запускать следующую ячейку.')

## 4. Полное восстановление данных и индекса
Скачивается PDF учебника, восстанавливаются текст и формулы всех 147 страниц,
пересоздаются dev-вопросы, эталоны и 8 авторских вопросов. Затем скачиваются
публичные веса Qwen3.5-4B Q4_K_M (~2.74 GB) и токенизатор; строится индекс.
LLM при подготовке данных и индекса не вызывается. QA не входят в корпус.

In [ ]:
import subprocess

result = subprocess.run(
    [PY, '-u', str(ROOT/'run.py'), 'prepare_all'],
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(result.stdout)
print('Код завершения:', result.returncode)

if result.returncode == 0:
    run('-m', 'unittest', 'discover', '-s', 'tests', '-v')
    print(json.loads((ROOT / 'index/manifest.json').read_text()))
else:
    result.check_returncode()

## 5. Один пробный вопрос


In [ ]:
if not RESUME:
    run('run_local_experiments.py', '--gpu-layers', -1 if GPU else 0,
        '--threads', min(8, os.cpu_count() or 2), '--smoke-test')
else:
    print('Продолжение: отдельный пробный вопрос пропущен.')


## 6. Все пять экспериментов
Видны номер режима, вопрос,
число вызовов, входные/выходные токены и время.
Сервер сбрасывает состояние между запросами; параметры Qwen одинаковы.

RESUME=False начинает новый расчёт. RESUME=True продолжает только тот же
прерванный расчёт с неизменными кодом, вопросами, индексом и runtime.
Расхождение журнала с сохранёнными trace блокирует продолжение.

In [ ]:
args = ['run_local_experiments.py', '--gpu-layers', -1 if GPU else 0,
        '--threads', min(8, os.cpu_count() or 2)]
if RESUME:
    args.append('--resume')
run(*args)
print((ROOT/'experiments.csv').read_text())


## 7. Отчёт


In [ ]:
run('verify_usage.py')
run('finalize_results.py')
run('validate_submission.py')
run('package_submission.py')
run('package_review.py')
run('checkpoint_run.py')
print('Слов в отчёте:', len((ROOT/'report.md').read_text().split()))
print((ROOT/'experiments.csv').read_text())
print('Готово: submission.zip и results_for_review.zip; проверьте математические объяснения.')


## 8. Скачивание


In [ ]:
if BASE == Path('/content'):
    from google.colab import files
    files.download(str(ROOT/'submission.zip'))
    files.download(str(ROOT/'results_for_review.zip'))
else:
    from IPython.display import display, FileLink
    display(FileLink(str(ROOT/'submission.zip')))
    display(FileLink(str(ROOT/'results_for_review.zip')))
    print('Также доступны в Output / Files:', ROOT)


In [ ]:
from pathlib import Path
from google.colab import files
import os, subprocess

backup_dir = Path('/content/rag_final_backup')
subprocess.run(
    [PY, '-u', str(ROOT/'run.py'), 'checkpoint_run'],
    cwd=ROOT,
    env={**os.environ, 'RAG_CHECKPOINT_DIR': str(backup_dir)},
    check=True
)
files.download(str(backup_dir / 'RAG_TFKP_checkpoint.zip'))